### 직렬화와 역직렬화로 모델 저장 및 로드하기
- 내가 만든 체인을 저장해야할 때 특정한 파일 확장자로 지정하기 어려우니 체인을 직렬화해서 JSON 형식으로 변환하여 저장

In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv() # .env 파일 로드
logging.langsmith("CH04-Models2") # LangSmith에 로깅 시작

LangSmith 추적을 시작합니다.
[프로젝트명]
CH04-Models2


In [5]:
prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?") # 프롬프트 템플릿 생성

In [ ]:
print(f"ChatOpenAI: {ChatOpenAI.is_lc_serializable()}") # ChatOpenAI가 직렬화 가능한지 확인

ChatOpenAI: True


In [7]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0) # ChatOpenAI 모델 생성

print(f"ChatOpenAI: {llm.is_lc_serializable()}") # ChatOpenAI 인스턴스가 직렬화 가능한지 확인

ChatOpenAI: True


In [8]:
chain = prompt | llm # 프롬프트와 LLM을 연결하여 체인 생성

chain.is_lc_serializable() # 체인이 직렬화 가능한지 확인

True

체인 직렬화하기

In [10]:
from langchain_core.load import dumpd, dumps

dumpd_chain = dumpd(chain) # 체인을 직렬화하여 딕셔너리로 변환
dumpd_chain # 직렬화된 체인 출력

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-3.5-turbo',
    'temperature': 0.0,
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

In [11]:
type(dumpd_chain) # 직렬화된 체인의 타입 확인

dict

In [12]:
dumps_chain = dumps(chain) # 체인을 직렬화하여 JSON 문자열로 변환
dumps_chain # 직렬화된 체인 출력

'{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "runnable", "RunnableSequence"], "kwargs": {"first": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["fruit"], "template": "{fruit}\\uc758 \\uc0c9\\uc0c1\\uc774 \\ubb34\\uc5c7\\uc785\\ub2c8\\uae4c?", "template_format": "f-string"}, "name": "PromptTemplate"}, "last": {"lc": 1, "type": "constructor", "id": ["langchain", "chat_models", "openai", "ChatOpenAI"], "kwargs": {"model_name": "gpt-3.5-turbo", "temperature": 0.0, "openai_api_key": {"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}, "stream_usage": true}, "name": "ChatOpenAI"}}, "name": "RunnableSequence"}'

In [13]:
type(dumps_chain) # 직렬화된 체인의 타입 확인

str

Pickle 파일로 직렬화하고 로드하기

In [ ]:
import pickle # 체인을 직렬화하여 바이트로 변환

# fruit_chain.pkl 파일로 직렬화된 체인을 저장
with open("fruit_chain.pkl", "wb") as f: # with 문이 open하고 close까지 처리
    pickle.dump(dumpd_chain, f) # pickle.dump() 함수를 사용하여 딕셔너리를 바이트 형식으로 저장

In [17]:
import json

with open("fruit_chain.json", "w") as fp: # fruit_chain.json 파일로 직렬화된 체인을 저장
    json.dump(dumpd_chain, fp) # json.dump() 함수를 사용하여 딕셔너리를 JSON 형식으로 저장

In [ ]:
import pickle

with open("fruit_chain.pkl", "rb") as f: # 피클파일인 fruit_chain.pkl 파일에서 직렬화된 체인을 읽어옴
    loaded_chain = pickle.load(f) # pickle.load() 함수를 사용하여 바이트 형식의 데이터를 딕셔너리로 변환

In [ ]:
from langchain_core.load import load # 직렬화된 체인을 역직렬화하여 체인 객체로 변환

chain_from_file = load(loaded_chain, allowed_objects="all") # 체인을 로드, allowed_objects="all" 옵션을 사용하여 모든 객체를 허용

print(chain_from_file.invoke({"fruit": "사과"})) # 체인을 실행하여 사과의 색상을 출력

content='사과의 색상은 주로 빨간색이지만, 녹색, 노란색, 주황색 등 다양한 색상의 사과도 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 24, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EOylA1jaVkNODDweVG1U8Zu8EeNVO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0adca-b904-7120-83cc-bf4a0f83d9f0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 51, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [21]:
from langchain_core.load import load, loads # 직렬화된 체인을 역직렬화하여 체인 객체로 변환

load_chain = load( # 체인을 로드, secrets_map={"OPENAI_API_KEY": os.environ.get("OPENAI_API_KEY")} 옵션을 사용하여 환경 변수에서 API 키를 가져옴
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ.get("OPENAI_API_KEY")}, allowed_objects="all"
)

load_chain.invoke({"fruit": "사과"}) # 체인을 실행하여 사과의 색상을 출력


AIMessage(content='사과의 색상은 주로 빨간색이지만, 녹색, 노란색, 주황색 등 다양한 색상의 사과도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 24, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EOypI5VwyzO5i8XxtYwmoDCyL3OdL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0adce-a417-7743-a9cf-eb15b5983bc8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 51, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [23]:
with open("fruit_chain.json", "r") as fp: # fruit_chain.json 파일에서 직렬화된 체인을 읽어옴
    loaded_chain_json = json.load(fp) # json.load() 함수를 사용하여 JSON 형식의 데이터를 딕셔너리로 변환
    loads_chain = load(loaded_chain_json, allowed_objects="all") # 체인을 로드

loads_chain.invoke({"fruit": "사과"}) # 체인을 실행하여 사과의 색상을 출력

AIMessage(content='사과의 색상은 주로 빨간색이지만, 녹색, 노란색, 주황색 등 다양한 색상의 사과도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 24, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EOyt4ZCaHXQcjMIAEzLN6kDhAU2bd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0add2-3635-72e2-af89-00b86016950c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 51, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})